# 3 · GRPO / RLVR (GPU DDP job)
Trains on top of the SFT model with the **verifiable reward** (format + name + param-name + param-value); uploads the merged model to `{grpo_model_s3}`. Entry point **`grpo.sh`** → `torchrun ... grpo_job.py`. Set `use_vllm=1` in notebook 0 (needs vLLM in the image) to speed up rollouts.

In [ ]:
import json
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, StoppingCondition

sess = Session()
cfg = json.load(open('toolcall_config.json'))
region = cfg['region']
INSTANCE_TYPE = cfg['gpu_instance']
image_uri = cfg['gpu_image_uri'] or image_uris.retrieve(
    framework='pytorch', region=region, version=cfg['pt_version'],
    instance_type=INSTANCE_TYPE, image_scope='training')
print('Region  :', region)
print('Instance:', INSTANCE_TYPE)
print('Image   :', image_uri)

In [ ]:
environment = {
    'BASE_MODEL': cfg['base_model'],
    'SFT_MODEL_S3': cfg['sft_model_s3'],
    'DATA_S3': cfg['data_s3'],
    'OUTPUT_GRPO_S3': cfg['grpo_model_s3'],
    'SEQ_LEN': str(cfg['seq_len']), 'SEED': str(cfg['seed']), 'ATTN_IMPL': cfg['attn_impl'],
    'USE_LORA': str(cfg['use_lora']), 'LORA_R': str(cfg['lora_r']), 'LORA_ALPHA': str(cfg['lora_alpha']),
    'GRPO_EPOCHS': str(cfg['grpo_epochs']), 'GRPO_LR': str(cfg['grpo_lr']),
    'GRPO_BS': str(cfg['grpo_bs']), 'GRPO_GRAD_ACCUM': str(cfg['grpo_grad_accum']),
    'NUM_GENERATIONS': str(cfg['num_generations']), 'GRPO_BETA': str(cfg['grpo_beta']),
    'GRPO_TEMPERATURE': str(cfg['grpo_temperature']),
    'MAX_COMPLETION_LENGTH': str(cfg['max_completion_length']),
    'REWARD_WEIGHTS': cfg['reward_weights'], 'USE_VLLM': str(cfg['use_vllm']),
    'NUM_GPUS': str(cfg['num_gpus']),
    'HF_HOME': '/opt/ml/hf_home', 'HF_HUB_ENABLE_HF_TRANSFER': '1',
    'TOKENIZERS_PARALLELISM': 'false', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
}
if cfg.get('hf_token'):
    environment['HF_TOKEN'] = cfg['hf_token']
    environment['HUGGING_FACE_HUB_TOKEN'] = cfg['hf_token']
source_code = SourceCode(source_dir='../src', requirements='requirements.txt', entry_script='grpo.sh')
trainer = ModelTrainer(sagemaker_session=sess, role=cfg['role'], training_image=image_uri,
    source_code=source_code,
    compute=Compute(instance_type=INSTANCE_TYPE, instance_count=1, keep_alive_period_in_seconds=3600),
    environment=environment, base_job_name='toolcall-grpo',
    stopping_condition=StoppingCondition(max_runtime_in_seconds=24*3600))
trainer.train(wait=True)
print('SFT+GRPO model ->', cfg['grpo_model_s3'])

Watch the per-component reward curves (`reward_format`, `reward_name`, `reward_param_name`, `reward_param_value`) to see what RL is improving.

**Next:** `04_evaluate.ipynb`